# Test Benchmark — Mini protocole

Protocole réduit pour valider le pipeline avant le benchmark complet.

| Paramètre | Valeur |
|---|---|
| Images profiling | 30 (640×640) |
| Images MAP | 10 (résolution originale) |
| Warmup | 5 itérations |
| Mesure | 10 itérations |
| Seed | 42 |

**Deux modes distincts :**
- **Profiling** → `load_model()` + `preprocess()` : images 640×640, comparaison de vitesse standardisée
- **MAP eval** → `load_model_eval()` + `evaluate_map()` : résolution native du modèle, conforme COCO

## 1. Configuration

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

import torch
import numpy as np
import pandas as pd
from pathlib import Path

IMG_DIR     = "datasets/coco/val2017"
ANN_FILE    = "datasets/coco/annotations/instances_val2017.json"
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

N_PROFILE_IMAGES = 300
N_EVAL_IMAGES    = 10
N_WARMUP         = 5
N_MEASURE        = 10
SEED             = 42
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : NVIDIA GeForce RTX 5060 Laptop GPU
VRAM   : 8.5 GB


## 2. Chargement des données

- `load_profiling_data` → images redimensionnées à 640×640
- `load_eval_data` → images à résolution originale COCO (variable)

In [2]:
from utils.data_loader import load_profiling_data, load_eval_data

profile_data        = load_profiling_data(IMG_DIR, ANN_FILE, n=N_PROFILE_IMAGES, seed=SEED)
eval_data, coco_gt  = load_eval_data(IMG_DIR, ANN_FILE, n=N_EVAL_IMAGES, seed=SEED)

print(profile_data)           # LazySampleList(n=30, ...)
print(f"Premier sample profiling : {profile_data[0]}")
print(f"Premier sample eval      : {eval_data[0]}")

loading annotations into memory...
Done (t=0.49s)
creating index...
index created!
loading annotations into memory...
Done (t=0.42s)
creating index...
index created!
LazySampleList(n=300, img_dir='datasets/coco/val2017')
Premier sample profiling : {'image_id': 107554, 'path': 'datasets\\coco\\val2017\\000000107554.jpg', 'orig_size': (480, 640)}
Premier sample eval      : {'image_id': 107554, 'path': 'datasets\\coco\\val2017\\000000107554.jpg', 'orig_size': (480, 640)}


## 3. Fonctions utilitaires

In [3]:
from utils.profiler import profile_model

# Accumulateurs pour les CSV finaux
profiling_rows, map_rows = [], []

def record(name, prof, maps):
    profiling_rows.append({"model": name, **prof})
    map_rows.append({"model": name, **maps})
    print(f"  Profiling (640×640) : {prof['mean_ms']:.1f} ms ± {prof['std_ms']:.1f}  ({prof['fps']:.1f} FPS)")
    print(f"  MAP (native res)    : AP={maps.get('AP', -1):.3f}  AP50={maps.get('AP50', -1):.3f}")

print("OK — profile_model importé depuis utils/profiler.py")

OK — profile_model importé depuis utils/profiler.py


## 4. RetinaNet R50

In [4]:
import models.retinanet_r50 as r50

print("Chargement profiling model...")
model_r50 = r50.load_model(DEVICE)
prof_r50  = profile_model(model_r50, profile_data, r50.preprocess, r50.collate,
                          n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_r50; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_r50_eval = r50.load_model_eval(DEVICE)
map_r50        = r50.evaluate_map(model_r50_eval, eval_data, coco_gt, DEVICE)
del model_r50_eval; torch.cuda.empty_cache()

record("retinanet_r50", prof_r50, map_r50)

Chargement profiling model...
Chargement eval model + MAP...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.12s).
Accumulating evaluation results...
DONE (t=0.08s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.525
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.820
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.535
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.305
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.546
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.728
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.428
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.537
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.573
 Averag

## 5. RetinaNet R101

In [5]:
import models.retinanet_r101 as r101

print("Chargement profiling model...")
model_r101 = r101.load_model(DEVICE)
prof_r101  = profile_model(model_r101, profile_data, r101.preprocess, r101.collate,
                           n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_r101; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_r101_eval = r101.load_model_eval(DEVICE)
map_r101        = r101.evaluate_map(model_r101_eval, eval_data, coco_gt, DEVICE)
del model_r101_eval; torch.cuda.empty_cache()

record("retinanet_r101", prof_r101, map_r101)

Loading config d:\school\ProjectDSAI2026_2\detectron2\model_zoo\configs\COCO-Detection\../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


Chargement profiling model...


The checkpoint state_dict contains keys that are not used by the model:
  pixel_mean
  pixel_std
c:\ProgramData\anaconda3\Lib\site-packages\torch\functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4324.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Loading config d:\school\ProjectDSAI2026_2\detectron2\model_zoo\configs\COCO-Detection\../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


Chargement eval model + MAP...


The checkpoint state_dict contains keys that are not used by the model:
  pixel_mean
  pixel_std


Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.06s).
Accumulating evaluation results...
DONE (t=0.06s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.474
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.708
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.494
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.203
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.520
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.708
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.371
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.519
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.573
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10

## 6. FCOS R50

In [6]:
import models.fcos_r50 as fcos

print("Chargement profiling model...")
model_fcos = fcos.load_model(DEVICE)
prof_fcos  = profile_model(model_fcos, profile_data, fcos.preprocess, fcos.collate,
                           n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_fcos; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_fcos_eval = fcos.load_model_eval(DEVICE)
map_fcos        = fcos.evaluate_map(model_fcos_eval, eval_data, coco_gt, DEVICE)
del model_fcos_eval; torch.cuda.empty_cache()

record("fcos_r50", prof_fcos, map_fcos)

Chargement profiling model...
Chargement eval model + MAP...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.43s).
Accumulating evaluation results...
DONE (t=0.30s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.514
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.764
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.580
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.366
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.533
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.674
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.389
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.541
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.569
 Averag

## 7. EfficientDet — test pipeline (PIL + F.normalize, sans +1)

In [7]:
# ── Test rapide D4 avec le pipeline PIL copié de l'ancien code ──────────────────
import importlib
import models.efficientdet_d4 as ed4
importlib.reload(ed4)

print("Chargement D4 eval model...")
model_ed4_test = ed4.load_model_eval(DEVICE)

print(f"Évaluation MAP sur {N_EVAL_IMAGES} images...")
map_ed4_test = ed4.evaluate_map(model_ed4_test, eval_data, coco_gt, DEVICE)
del model_ed4_test; torch.cuda.empty_cache()

print(f"\n>>> D4 AP  = {map_ed4_test['AP']:.4f}  (attendu ~0.43)")
print(f">>> D4 AP50= {map_ed4_test['AP50']:.4f}  (attendu ~0.62)")
print(f"\nRésultat complet : {map_ed4_test}")

Chargement D4 eval model...
Évaluation MAP sur 10 images...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.05s).
Accumulating evaluation results...
DONE (t=0.06s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.582
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.738
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.633
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.362
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.593
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.879
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.459
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.618
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.648
 Average

In [8]:
import models.efficientdet_d4 as ed4

print("Chargement profiling model...")
model_ed4 = ed4.load_model(DEVICE)
prof_ed4  = profile_model(model_ed4, profile_data, ed4.preprocess, ed4.collate,
                          n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_ed4; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_ed4_eval = ed4.load_model_eval(DEVICE)
map_ed4        = ed4.evaluate_map(model_ed4_eval, eval_data, coco_gt, DEVICE)
del model_ed4_eval; torch.cuda.empty_cache()

record("efficientdet_d4", prof_ed4, map_ed4)

Chargement profiling model...
Chargement eval model + MAP...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.18s).
Accumulating evaluation results...
DONE (t=0.23s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.582
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.738
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.633
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.362
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.593
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.879
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.459
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.618
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.648
 Averag

## 8. EfficientDet D5

In [9]:
import models.efficientdet_d5 as ed5

print("Chargement profiling model...")
model_ed5 = ed5.load_model(DEVICE)
prof_ed5  = profile_model(model_ed5, profile_data, ed5.preprocess, ed5.collate,
                          n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_ed5; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_ed5_eval = ed5.load_model_eval(DEVICE)
map_ed5        = ed5.evaluate_map(model_ed5_eval, eval_data, coco_gt, DEVICE)
del model_ed5_eval; torch.cuda.empty_cache()

record("efficientdet_d5", prof_ed5, map_ed5)

Chargement profiling model...
Chargement eval model + MAP...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.06s).
Accumulating evaluation results...
DONE (t=0.06s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.524
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.728
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.605
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.417
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.573
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.764
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.415
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.555
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.587
 Averag

## 9. EfficientDet D6

In [10]:
import models.efficientdet_d6 as ed6

print("Chargement profiling model...")
model_ed6 = ed6.load_model(DEVICE)
prof_ed6  = profile_model(model_ed6, profile_data, ed6.preprocess, ed6.collate,
                          n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
del model_ed6; torch.cuda.empty_cache()

print("Chargement eval model + MAP...")
model_ed6_eval = ed6.load_model_eval(DEVICE)
map_ed6        = ed6.evaluate_map(model_ed6_eval, eval_data, coco_gt, DEVICE)
del model_ed6_eval; torch.cuda.empty_cache()

record("efficientdet_d6", prof_ed6, map_ed6)

Chargement profiling model...
Chargement eval model + MAP...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.09s).
Accumulating evaluation results...
DONE (t=0.10s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.507
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.692
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.532
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.295
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.561
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.760
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.374
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.584
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.620
 Averag

## 10. Résultats

In [11]:
df_prof = pd.DataFrame(profiling_rows).set_index("model")
df_map  = pd.DataFrame(map_rows).set_index("model")

df_prof.to_csv(RESULTS_DIR / "profiling_test.csv")
df_map.to_csv(RESULTS_DIR  / "map_test.csv")

print("=== Profiling — 640×640 (ms) ===")
display(df_prof.round(2))

print("\n=== MAP — résolution native COCO ===")
display(df_map.round(3))

print(f"\nCSV sauvegardés dans {RESULTS_DIR.resolve()}")

=== Profiling — 640×640 (ms) ===


,mean_ms,std_ms,min_ms,max_ms,fps
model,,,,,
retinanet_r50,63.21,1.82,59.81,66.36,15.82
retinanet_r101,75.66,7.09,69.53,94.95,13.22
fcos_r50,60.71,6.20,55.04,77.93,16.47
efficientdet_d4,67.10,7.83,57.07,85.81,14.90
efficientdet_d5,65.00,3.42,60.61,71.66,15.39
efficientdet_d6,82.40,13.07,76.21,121.49,12.14



=== MAP — résolution native COCO ===


,AP,AP50,AP75,APs,APm,APl
model,,,,,,
retinanet_r50,0.525,0.820,0.535,0.305,0.546,0.728
retinanet_r101,0.474,0.708,0.494,0.203,0.520,0.708
fcos_r50,0.514,0.764,0.580,0.366,0.533,0.674
efficientdet_d4,0.582,0.738,0.633,0.362,0.593,0.879
efficientdet_d5,0.524,0.728,0.605,0.417,0.573,0.764
efficientdet_d6,0.507,0.692,0.532,0.295,0.561,0.760



CSV sauvegardés dans D:\school\ProjectDSAI2026_2\results


## 11. Profilage méso-scopique — PyTorch Profiler

Décomposition du forward pass par opération PyTorch (couche par couche).

Le profiler utilise son propre `schedule(wait=0, warmup=N_WARMUP, active=N_MEASURE)` :
- phase warmup : GPU chauffe, trace collectée mais non commitée
- phase active : trace exportée vers TensorBoard + Chrome JSON

**Sorties :**
```
results/profiler/pytorch/<model>/
  tensorboard/        ← tensorboard --logdir results/profiler/pytorch/<model>/tensorboard
  chrome_trace.json   ← chrome://tracing  ou  ui.perfetto.dev
  summary*.txt        ← tableaux triés par cuda_time_total
```

In [12]:
from profiler.pytorch_profiler import profile_with_pytorch

PROF_OUTPUT  = "results/profiler/pytorch"
PROFILER_TAG = "baseline"   # modifier ici pour les runs suivants : "tensorRt", "compiled", etc.

MODELS_TO_PROFILE = {
    "retinanet_r50":   (r50,  r50.load_model),
    "retinanet_r101":  (r101, r101.load_model),
    "fcos_r50":        (fcos, fcos.load_model),
    "efficientdet_d4": (ed4,  ed4.load_model),
    "efficientdet_d5": (ed5,  ed5.load_model),
    "efficientdet_d6": (ed6,  ed6.load_model),
}

pytorch_prof_results = {}

for model_name, (mod, load_fn) in MODELS_TO_PROFILE.items():
    print(f"\n{'─'*55}")
    print(f"  Profiling : {model_name}  [tag={PROFILER_TAG}]")
    print(f"{'─'*55}")

    model = load_fn(DEVICE)

    result = profile_with_pytorch(
        model         = model,
        data          = profile_data,
        preprocess_fn = mod.preprocess,
        collate_fn    = mod.collate,
        n_warmup      = N_WARMUP,
        n_active      = N_MEASURE,
        output_dir    = PROF_OUTPUT,
        model_name    = model_name,
        tag           = PROFILER_TAG,
        device        = DEVICE,
    )

    pytorch_prof_results[model_name] = result
    del model
    torch.cuda.empty_cache()

print(f"\n✓ Profiling terminé.")
print(f"  TensorBoard : tensorboard --logdir {PROF_OUTPUT}")


───────────────────────────────────────────────────────
  Profiling : retinanet_r50  [tag=baseline]
───────────────────────────────────────────────────────

  PyTorch Profiler — retinanet_r50—baseline—20260610_002234
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  Total KFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  -----


───────────────────────────────────────────────────────
  Profiling : retinanet_r101  [tag=baseline]
───────────────────────────────────────────────────────


  pixel_mean
  pixel_std



  PyTorch Profiler — retinanet_r101—baseline—20260610_002305
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  Total KFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          model_forward         0.00%       0.000us         0.00%       0.000us       0.000us     86

## 12. Profilage feuille par feuille — `profile_model_detailed`

Décomposition du forward pass **module atomique par module atomique** (Conv2d, BatchNorm2d, ReLU, etc.) via des CUDA Events attachés en hooks.

**Principe :**
- `forward_pre_hook` → `start.record()` injecté dans le stream GPU avant chaque feuille
- `forward_hook` → `end.record()` injecté après chaque feuille
- `elapsed_time(start, end)` lu après `synchronize()` = temps GPU réel en ms

**Sortie :** un DataFrame avec une ligne par feuille, trié par `mean_ms` décroissant.

> **Note :** `sum(leaves.mean_ms) ≥ global.mean_ms` car CUDA peut exécuter certains kernels en parallèle. L'écart entre les deux révèle le taux de parallélisme interne du modèle.

In [13]:
from utils.profiler import profile_model_detailed

# ── Choix du modèle à décomposer ──────────────────────────────────────────────
# Changer ici pour analyser un autre modèle
TARGET_MODEL_NAME = "retinanet_r50"
TARGET_MOD        = r50
TARGET_LOAD_FN    = r50.load_model

print(f"Profilage feuille par feuille : {TARGET_MODEL_NAME}")
print(f"  warmup={N_WARMUP} itérations, mesure={N_MEASURE} itérations\n")

model_leaf = TARGET_LOAD_FN(DEVICE)

detailed = profile_model_detailed(
    model         = model_leaf,
    data          = profile_data,
    preprocess_fn = TARGET_MOD.preprocess,
    collate_fn    = TARGET_MOD.collate,
    n_warmup      = N_WARMUP,
    n_measure     = N_MEASURE,
    device        = DEVICE,
)

del model_leaf; torch.cuda.empty_cache()

# ── Résultats globaux ─────────────────────────────────────────────────────────
g = detailed["global"]
print(f"{'─'*55}")
print(f"  Global  : {g['mean_ms']:.2f} ms ± {g['std_ms']:.2f}  ({g['fps']:.1f} FPS)")

df_leaves = detailed["leaves"]
sum_leaves = df_leaves["mean_ms"].sum()
print(f"  Σ feuilles : {sum_leaves:.2f} ms  (parallélisme GPU = {sum_leaves/g['mean_ms']:.2f}x)")
print(f"  Nombre de feuilles actives : {len(df_leaves)}")
print(f"{'─'*55}\n")

# ── Top 20 feuilles les plus coûteuses ────────────────────────────────────────
print("Top 20 feuilles (mean_ms décroissant) :")
display(df_leaves.head(20)[["module", "type", "component", "mean_ms", "std_ms", "pct_sum", "n"]].round(4))

# ── Agrégation par composant ──────────────────────────────────────────────────
print("\nTemps par composant (backbone, fpn, head, …) :")
by_component = (df_leaves.groupby("component")["mean_ms"]
                .sum()
                .sort_values(ascending=False)
                .reset_index())
by_component["pct_total"] = (by_component["mean_ms"] / sum_leaves * 100).round(2)
display(by_component.round(4))

Profilage feuille par feuille : retinanet_r50
  warmup=5 itérations, mesure=10 itérations

───────────────────────────────────────────────────────
  Global  : 77.20 ms ± 12.68  (13.0 FPS)
  Σ feuilles : 27.69 ms  (parallélisme GPU = 0.36x)
  Nombre de feuilles actives : 160
───────────────────────────────────────────────────────

Top 20 feuilles (mean_ms décroissant) :


,module,type,component,mean_ms,std_ms,pct_sum,n
0,backbone.fpn.layer_blocks.0.0,0,backbone,1.3901,0.0210,5.02,10
1,transform,transform,transform,0.9407,0.3945,3.40,10
2,backbone.body.conv1,conv1,backbone,0.9345,0.0710,3.37,10
3,anchor_generator,anchor_generator,anchor_generator,0.7415,0.6247,2.68,10
4,backbone.body.layer4.0.downsample.0,0,backbone,0.6927,0.0507,2.50,10
5,backbone.body.layer2.0.downsample.0,0,backbone,0.6706,0.0225,2.42,10
6,backbone.body.layer3.0.downsample.0,0,backbone,0.6582,0.0471,2.38,10
7,backbone.body.layer4.0.conv2,conv2,backbone,0.6433,0.0514,2.32,10
8,backbone.body.layer4.1.conv2,conv2,backbone,0.5840,0.0806,2.11,10
9,backbone.body.layer4.2.conv2,conv2,backbone,0.5708,0.0376,2.06,10



Temps par composant (backbone, fpn, head, …) :


,component,mean_ms,pct_total
0,backbone,23.6701,85.48
1,head,2.3385,8.45
2,transform,0.9407,3.40
3,anchor_generator,0.7415,2.68


In [14]:
# ── Décomposition feuille sur tous les modèles ──────────────────────────────
# Cellule optionnelle — peut prendre du temps (6 modèles × N_MEASURE itérations)

MODELS_TO_DETAIL = {
    "retinanet_r50":   (r50,  r50.load_model),
    "retinanet_r101":  (r101, r101.load_model),
    "fcos_r50":        (fcos, fcos.load_model),
    "efficientdet_d4": (ed4,  ed4.load_model),
    "efficientdet_d5": (ed5,  ed5.load_model),
    "efficientdet_d6": (ed6,  ed6.load_model),
}

all_leaves   = {}   # model_name → DataFrame feuilles
all_globals  = {}   # model_name → dict stats global

for mname, (mod, load_fn) in MODELS_TO_DETAIL.items():
    print(f"  {mname}…", end=" ", flush=True)
    m = load_fn(DEVICE)
    res = profile_model_detailed(m, profile_data, mod.preprocess, mod.collate,
                                 n_warmup=N_WARMUP, n_measure=N_MEASURE, device=DEVICE)
    all_leaves[mname]  = res["leaves"]
    all_globals[mname] = res["global"]
    del m; torch.cuda.empty_cache()
    g = res["global"]
    print(f"{g['mean_ms']:.1f} ms  ({g['fps']:.1f} FPS)  — {len(res['leaves'])} feuilles")

# ── Tableau récapitulatif : top-5 feuilles par modèle ─────────────────────────
print("\n\n=== Top 5 feuilles les plus lentes par modèle ===")
for mname, df in all_leaves.items():
    print(f"\n{mname} (global={all_globals[mname]['mean_ms']:.1f} ms):")
    display(df.head(5)[["module", "mean_ms", "pct_sum"]].round(4))

  retinanet_r50… 67.6 ms  (14.8 FPS)  — 160 feuilles
  retinanet_r101… 

  pixel_mean
  pixel_std


ValueError: Both events must be recorded before calculating elapsed time.